
# Single-Cell Best Practices – 10-Day Mini Bootcamp

Welcome! This friendly notebook translates the topics you listed into a 10-day practice plan. Each day mirrors the structure used in the MateenahJAHAN/Deep--TCR curriculum: we keep the language super simple, focus on the exact skill, and give you runnable starter code.

> 📎 Reference inspiration: https://www.sc-best-practices.org/introduction/prior_art.html (I cannot open the site directly from this environment, so the steps below are based on the widely adopted workflows that guide covers.)



## How to use this notebook
- Treat each day as a mini lesson (takes 30–60 minutes).
- Read the short summary, then run the code cell right under it.
- Replace or extend the starter code with your own experiments.
- Keep notes inside the markdown blocks (double-click to edit in Jupyter).
- If a package is missing, run the optional `%pip install` cell first.



## Demo data & packages
We will lean on Scanpy's `pbmc3k` dataset so you can run everything locally without hunting for files. You can swap it with your own AnnData object later.


In [ ]:

# Optional: install everything in one go (uncomment the next line if needed)
# %pip install --quiet scanpy scvi-tools scvelo gseapy networkx


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

try:
    import scvi
except ImportError:
    scvi = None
    print("⚠️ Install scvi-tools (pip install scvi-tools) to unlock the perturbation modeling demo.")

try:
    import gseapy as gp
except ImportError:
    gp = None
    print("⚠️ Install gseapy (pip install gseapy) to run the GSEA cell.")

try:
    import scvelo as scv
except ImportError:
    scv = None
    print("⚠️ Install scvelo (pip install scvelo) to run the RNA velocity cell.")

import networkx as nx

sc.settings.verbosity = 0
sc.set_figure_params(dpi=100)


In [ ]:

adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()
adata.layers["counts"] = adata.X.copy()
adata.obs["source_dataset"] = "pbmc3k"
adata.raw = adata
print(f"Cells: {adata.n_obs:,} | Genes: {adata.n_vars:,}")


In [ ]:

adatainfo = (
    adata.obs[["n_genes", "total_counts"]]
    .describe()
    .rename(index={"50%": "median"})
)
adatainfo



## Day 01 – Introduction, Pre-processing & Visualization
**Plain-language goal:** Give the raw matrix a spa day so every cell and gene is clean, normalized, and ready for downstream tricks.

**What we will do:**
1. Drop low-quality cells/genes.
2. Normalize counts, log-transform, and keep highly variable genes.
3. Run PCA + UMAP so we can *see* the data.

**Why this matters:** Everything else (clustering, DEG, velocity) expects tidy inputs. Garbage in → garbage out.


In [ ]:

adata_day1 = adata.copy()
print("Before QC:", adata_day1.shape)

sc.pp.filter_cells(adata_day1, min_genes=200)
sc.pp.filter_genes(adata_day1, min_cells=3)
adata_day1.var["mt"] = adata_day1.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata_day1, qc_vars=["mt"], inplace=True)
adata_day1 = adata_day1[adata_day1.obs["pct_counts_mt"] < 15, :]

sc.pp.normalize_total(adata_day1, target_sum=1e4)
sc.pp.log1p(adata_day1)
sc.pp.highly_variable_genes(adata_day1, n_top_genes=2000, subset=True)
sc.pp.scale(adata_day1, max_value=10)

sc.tl.pca(adata_day1, n_comps=50)
sc.pp.neighbors(adata_day1, n_neighbors=15)
sc.tl.umap(adata_day1)

print("After QC:", adata_day1.shape)
sc.pl.umap(adata_day1, color=["n_genes", "pct_counts_mt"], frameon=False)



## Day 02 – Clustering, Annotation & Data Integration
**Goal:** Turn blobs on the UMAP into real cell types and make sure batches mix nicely.

**Steps:**
1. Cluster with Leiden.
2. Assign names using marker dictionaries (feel free to tweak!).
3. Simulate a second batch and integrate it to show the workflow (replace with your real batches later).


In [ ]:

adata_day2 = adata_day1.copy()
sc.tl.leiden(adata_day2, resolution=0.5, key_added="leiden")

marker_map = {
    "0": "Naive T",
    "1": "Memory T",
    "2": "B cell",
    "3": "NK",
    "4": "Myeloid",
    "5": "Plasma",
}
adata_day2.obs["cell_type"] = (
    adata_day2.obs["leiden"].map(marker_map).fillna("Other")
)

sc.pl.umap(adata_day2, color=["leiden", "cell_type"], frameon=False)

# --- Tiny integration demo (duplicate data to mimic a second batch) ---
adata_day2.obs["batch"] = "Batch_A"
adata_batch_b = adata_day2[:1000].copy()
adata_batch_b.obs["batch"] = "Batch_B"
adata_integrated = sc.concat([adata_day2, adata_batch_b], join="outer")

sc.pp.normalize_total(adata_integrated, target_sum=1e4)
sc.pp.log1p(adata_integrated)
sc.pp.highly_variable_genes(adata_integrated, n_top_genes=2000, subset=True)
sc.pp.scale(adata_integrated, max_value=10)
sc.tl.pca(adata_integrated)
sc.pp.neighbors(adata_integrated, n_neighbors=15)
sc.tl.umap(adata_integrated)
sc.pp.combat(adata_integrated, key="batch")

sc.pl.umap(adata_integrated, color=["batch"], title=["After simple integration"], frameon=False)



## Day 03 – DEG Analysis, GSEA & Pathways
**Goal:** Find marker genes that make each cluster unique, then see which pathways pop up.

**Steps:**
1. Differential expression with `sc.tl.rank_genes_groups`.
2. Pull the gene table into pandas.
3. (Optional) Send the top genes to GSEA/Enrichr for pathway stories.


In [ ]:

adata_day3 = adata_day2.copy()
sc.tl.rank_genes_groups(adata_day3, groupby="cell_type", method="wilcoxon")
sc.pl.rank_genes_groups(adata_day3, n_genes=5, sharey=False)

marker_df = sc.get.rank_genes_groups_df(adata_day3, group=None)
top5 = marker_df.groupby("group").head(5)
print(top5.head())

selected_group = adata_day3.obs["cell_type"].unique()[0]
top_genes = (
    marker_df[marker_df["group"] == selected_group]
    .sort_values("scores", ascending=False)
    .head(50)["names"].tolist()
)

if gp is None:
    print("Install gseapy to run enrichment analysis.")
else:
    enr = gp.enrichr(
        gene_list=top_genes,
        description=f"{selected_group}_signature",
        gene_sets="GO_Biological_Process_2023",
        outdir=None
    )
    display(enr.results.head())



## Day 04 – Perturbation Modelling
**Goal:** Compare control vs. treatment (or before vs. after) to see what changes. We simulate labels so you can swap in your real metadata later.

**Steps:**
1. Add a fake `condition` column.
2. Train a tiny logistic regression on the PCA features to predict the condition (this mirrors perturbation-response models like scGen/scVI).
3. Run DEG between conditions to spot perturbed genes.
4. (Bonus) If `scvi-tools` is installed, learn a latent space that respects batches + conditions.


In [ ]:

adata_day4 = adata_day2.copy()
np.random.seed(7)
adata_day4.obs["condition"] = np.random.choice([
    "control",
    "perturbation"
], size=adata_day4.n_obs, p=[0.6, 0.4])

X = adata_day4.obsm["X_pca"]
y = (adata_day4.obs["condition"] == "perturbation").astype(int).values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7
)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
print(classification_report(y_test, preds, target_names=["control", "perturbation"]))

sc.tl.rank_genes_groups(adata_day4, groupby="condition", method="wilcoxon")
sc.pl.rank_genes_groups(adata_day4, n_genes=5, sharey=False)

if scvi is not None:
    scvi.data.setup_anndata(adata_day4, batch_key="condition")
    vae = scvi.model.SCVI(adata_day4, n_latent=20)
    vae.train(max_epochs=30, plan_kwargs={"lr": 3e-3}, check_val_every_n_epoch=None)
    adata_day4.obsm["X_scvi"] = vae.get_latent_representation()
    print("Stored SCVI latent space in adata_day4.obsm['X_scvi']")
else:
    print("Install scvi-tools to learn perturbation-aware latent spaces like scVI/scANVI.")



## Day 05 – Gene Regulatory Networks (GRNs)
**Goal:** Peek at which transcription factors might control which genes. We build a toy Spearman-correlation network (fast + install-free) and point you toward pySCENIC for full workflows.

**Steps:**
1. Focus on a manageable set of genes (e.g., top 40 HVGs).
2. Compute pairwise correlations.
3. Build a NetworkX graph and inspect hubs.


In [ ]:

adata_day5 = adata_day4.copy()
key_genes = list(adata_day5.var_names[:40])
subset = adata_day5[:, key_genes].X
if hasattr(subset, "toarray"):
    subset = subset.toarray()
expr_df = pd.DataFrame(subset, columns=key_genes)

corr = expr_df.corr(method="spearman")
threshold = 0.45
G = nx.Graph()
G.add_nodes_from(key_genes)
for i, gene_i in enumerate(key_genes):
    for gene_j in key_genes[i + 1:]:
        weight = corr.loc[gene_i, gene_j]
        if abs(weight) >= threshold:
            G.add_edge(gene_i, gene_j, weight=float(weight))

print(f"Nodes: {G.number_of_nodes()} | Edges (|r| >= {threshold}): {G.number_of_edges()}")
degree_series = pd.Series(dict(G.degree())).sort_values(ascending=False)
degree_series.head(10)



## Day 06 – Pseudotemporal Ordering
**Goal:** Arrange cells along a timeline to mimic differentiation or activation paths.

**Steps:**
1. Reuse neighbors/UMAP from Day 1.
2. Compute a PAGA graph.
3. Pick a root population (here: Naive T).
4. Run diffusion pseudotime (DPT) and visualize.


In [ ]:

adata_day6 = adata_day2.copy()
sc.tl.paga(adata_day6, groups="cell_type")

root_mask = adata_day6.obs["cell_type"] == "Naive T"
if root_mask.sum() == 0:
    root_mask = np.zeros(adata_day6.n_obs, dtype=bool)
    root_mask[0] = True
adata_day6.uns["iroot"] = np.flatnonzero(root_mask)[0]

sc.tl.dpt(adata_day6)
sc.pl.paga(adata_day6, color=["cell_type"], title="PAGA abstract graph")
sc.pl.umap(adata_day6, color=["cell_type", "dpt_pseudotime"], frameon=False)



## Day 07 – Compositional Analysis & Cell–Cell Communication
**Goal:** Measure how cell-type proportions change between conditions and guess which ligands/receptors keep cells talking.

**Steps:**
1. Tabulate proportions per condition.
2. Compute simple ligand–receptor scores (average ligand in sender × average receptor in receiver).
3. Sort scores to highlight interesting interactions.


In [ ]:

adata_day7 = adata_day4.copy()
composition = (
    adata_day7.obs.groupby(["condition", "cell_type"])
    .size()
    .unstack(fill_value=0)
)
composition = composition.divide(composition.sum(axis=1), axis=0)
print("Cell-type fractions per condition:")
print(composition)

expr = adata_day7.to_df()
ligand_receptor_pairs = [
    ("CCL5", "CCR7"),
    ("IL7", "IL7R"),
    ("TNF", "TNFRSF1A"),
    ("CXCL13", "CXCR5")
]
cell_types = adata_day7.obs["cell_type"].unique()
rows = []
for ligand, receptor in ligand_receptor_pairs:
    if ligand not in expr.columns or receptor not in expr.columns:
        continue
    ligand_means = expr[ligand].groupby(adata_day7.obs["cell_type"]).mean()
    receptor_means = expr[receptor].groupby(adata_day7.obs["cell_type"]).mean()
    for sender in cell_types:
        for receiver in cell_types:
            score = ligand_means.get(sender, 0.0) * receptor_means.get(receiver, 0.0)
            rows.append({
                "ligand": ligand,
                "receptor": receptor,
                "sender": sender,
                "receiver": receiver,
                "score": score
            })

lr_scores = pd.DataFrame(rows)
if lr_scores.empty:
    print("No overlapping ligand/receptor genes found in this toy dataset.")
else:
    top_lr = lr_scores.sort_values("score", ascending=False).head(10)
    top_lr



## Day 08 – Bulk Deconvolution
**Goal:** Pretend we only have a bulk RNA-seq sample and estimate how many cells of each type are inside.

**Steps:**
1. Build cell-type reference profiles (average expression per type).
2. Create a fake bulk mixture using known fractions.
3. Solve a non-negative least squares problem to recover the fractions.


In [ ]:

adata_day8 = adata_day2.copy()
cell_type_profiles = (
    adata_day8.to_df()
    .groupby(adata_day8.obs["cell_type"])
    .mean()
)

seed_mix = pd.Series({
    "Naive T": 0.3,
    "Memory T": 0.25,
    "B cell": 0.2,
    "NK": 0.15,
    "Myeloid": 0.07,
    "Other": 0.03
})
seed_mix = seed_mix.reindex(cell_type_profiles.index, fill_value=0.05)
seed_mix = seed_mix / seed_mix.sum()

pseudo_bulk = cell_type_profiles.T @ seed_mix
library = cell_type_profiles.T.to_numpy()
target = pseudo_bulk.to_numpy()
fractions, _, _, _ = np.linalg.lstsq(library, target, rcond=None)
fractions = np.clip(fractions, 0, None)
fractions = fractions / fractions.sum()

estimated = pd.Series(fractions, index=cell_type_profiles.index)
result = pd.DataFrame({
    "true_fraction": seed_mix,
    "estimated_fraction": estimated
}).fillna(0)
result



## Day 09 – Lineage Tracing (Clonal Trees)
**Goal:** Track how clones expand. We simulate clone IDs, summarize frequencies, and draw a simple parent-child graph.

**Steps:**
1. Assign random clone IDs (replace with your CRISPR/barcode labels).
2. Count clone sizes.
3. Build a tiny lineage tree with NetworkX.


In [ ]:

adata_day9 = adata_day2.copy()
np.random.seed(11)
clone_ids = [f"clone_{i}" for i in range(1, 11)]
prob = np.linspace(1, 10, num=len(clone_ids))
prob = prob / prob.sum()
adata_day9.obs["clone_id"] = np.random.choice(clone_ids, size=adata_day9.n_obs, p=prob)

clone_sizes = adata_day9.obs["clone_id"].value_counts()
print("Clone sizes:")
print(clone_sizes)

G = nx.DiGraph()
for idx, clone in enumerate(clone_ids):
    G.add_node(clone, size=int(clone_sizes.get(clone, 0)))
    if idx == 0:
        continue
    parent = clone_ids[np.random.randint(0, idx)]
    G.add_edge(parent, clone)

print(f"Lineage edges: {G.number_of_edges()}")
print("Example edges:", list(G.edges())[:10])



## Day 10 – RNA Velocity
**Goal:** Predict where cells are heading by comparing spliced vs. unspliced RNA. We load the pancreas demo from scVelo so you can try the standard workflow.

**Steps:**
1. Download a velocity-ready dataset (`scv.datasets.pancreas()`).
2. Filter + normalize counts.
3. Compute velocities and the velocity graph.
4. Plot streamlines on UMAP.


In [ ]:

if scv is None:
    print("Install scvelo (pip install scvelo) to run RNA velocity.")
else:
    adata_velo = scv.datasets.pancreas()
    scv.pp.filter_and_normalize(adata_velo, min_shared_counts=20, n_top_genes=2000)
    scv.pp.moments(adata_velo, n_pcs=30, n_neighbors=30)
    scv.tl.velocity(adata_velo)
    scv.tl.velocity_graph(adata_velo)
    scv.pl.velocity_embedding_stream(adata_velo, basis="umap", color="clusters")



## Where to go next
- Swap the toy dataset with your real AnnData file and keep the same cell order.
- Replace the simulated metadata (batch, condition, clone) with true annotations.
- Consider saving intermediate `.h5ad` files (`adata_dayX.write('dayX.h5ad')`) to avoid recomputing.
- When you need production-strength versions of these steps, look up the corresponding chapters in the Single-Cell Best Practices book for deeper theory + tool comparisons.

🎉 Nice work! You now have a complete, beginner-friendly pipeline covering clustering, DEG/GSEA, perturbations, GRNs, pseudotime, composition, deconvolution, lineage, and RNA velocity.
